# Lesson 06: Creating Advanced Prompts

In this notebook, you will execute all 7 advanced prompting patterns with `google-genai`:
1. Few-Shot Shakespearean Sonnets
2. Chain-of-Thought (Lisa & Alice apple problem)
3. Generated Knowledge (`ACME Insurance` catalog)
4. Least-to-Most & Maieutic Prompting
5. Self-Refine Flask security upgrade + Temperature ($0.1$ vs $0.9$) comparison

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
cot_prompt = '''
Q: Lisa has 7 apples. She throws 1 away, gives 4 to Bart, and Bart gives 1 back. How many apples does Lisa have?
A: Lisa starts with 7. 7 - 1 = 6. 6 - 4 = 2. 2 + 1 = 3. Lisa has 3 apples.

Q: Alice has 5 apples. She throws 3 away, gives 2 to Bob, and Bob gives 1 back. How many apples does Alice have?
A:
'''

if client:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=cot_prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    print("Chain-of-Thought Result:\n", resp.text)
else:
    print("[Offline Demo Mode] Alice starts with 5. 5 - 3 = 2. 2 - 2 = 0. 0 + 1 = 1. Alice has 1 apple.")

In [ ]:
starter_flask = '''from flask import Flask, request
app = Flask(__name__)
@app.route('/')
def hello():
    name = request.args.get('name', 'World')
    return f'Hello, {name}!'
'''

if client:
    critique = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Suggest 3 security and architecture improvements for this Flask code, then rewrite it:\n```python\n{starter_flask}```",
    )
    print(critique.text)
else:
    print("[Offline Demo Mode] Improvements: 1. Escape name with markupsafe.escape. 2. Validate with FlaskForm. 3. Load FLASK_SECRET_KEY from os.environ.")